# Predictive Modeling Using Machine Learning

This notebook walks through preprocessing, Decision Tree and Random Forest classification, and model evaluation.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(ROOT / 'data' / 'customer_purchase_data.csv')
df.head()

## 1. Data preparation

In [ ]:
numeric_cols = ['age', 'annual_income', 'years_experience', 'website_visits', 'avg_session_minutes']
df = df.drop_duplicates().copy()
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())
X = df[numeric_cols]
y = df['purchased'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print('Train:', X_train.shape, 'Test:', X_test.shape)

## 2. Train Decision Tree and Random Forest

In [ ]:
models = {
    'Decision Tree': Pipeline([('scaler', StandardScaler()), ('classifier', DecisionTreeClassifier(max_depth=4, random_state=42))]),
    'Random Forest': Pipeline([('scaler', StandardScaler()), ('classifier', RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42))])
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, prob)
    })
results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
results_df

## 3. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, model) in zip(axes, models.items()):
    pred = model.predict(X_test)
    ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, cmap='Blues')
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 4. ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, model in models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name)
ax.plot([0, 1], [0, 1], linestyle='--', label='Random baseline')
ax.set_title('ROC Curve Comparison')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Feature importance

Random Forest importance shows which input variables contribute most to the classifier's decisions.

In [ ]:
rf = models['Random Forest'].named_steps['classifier']
importance = pd.Series(rf.feature_importances_, index=numeric_cols).sort_values(ascending=False)
sns.barplot(x=importance.values, y=importance.index)
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()